# Modified proxy: 32 shared trials

Select the RTX PRO 6000 GPU in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Run controls
RUN_SMOKE = True
RUN_FULL = False
# RUN_SMOKE trains one full trial (~1.33B tokens, 20 validations).
# RUN_FULL runs the 32-trial search and skips completed trials, including trial 000.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated"
DATA = WORKSPACE / "data" / "fineweb_edu"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([sys.executable, "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)

# Training runs in a subprocess of the same Python; tqdm renders in the notebook.
sys.path.insert(0, str(REPO / "src"))
from base_model_research.progress import run_with_progress

def training_data(*, single_proxy_trial=False):
    print(f"Checking the prepared dataset at {SAVED_DATA}…", flush=True)
    if not (SAVED_DATA / "manifest.json").is_file():
        raise FileNotFoundError(f"Run notebook 01 to prepare the shared dataset at {SAVED_DATA}.")
    if SAVED_DATA == DATA:
        return ["--data", SAVED_DATA]
    if not (DATA / "manifest.json").is_file():
        from inspect import signature
        from base_model_research.data import TokenCorpus
        if "cache_dir" in signature(TokenCorpus).parameters:
            print("Training will cache each shard on the runtime SSD when it is first read.", flush=True)
            return ["--data", SAVED_DATA, "--cache-dir", DATA.parent / "cached_shards"]
        if single_proxy_trial:
            print("Reading the shards for trial 000 directly from Google Drive.", flush=True)
            return ["--data", SAVED_DATA]
        print("This experiment's saved code copies the full dataset (about 48 GB) "
              "from Drive to the runtime SSD before training.", flush=True)
        print(f"To check the copy in the Colab Terminal: du -sh '{DATA}.copying'", flush=True)
    else:
        print("Checking the prepared copy on the runtime SSD…", flush=True)
    run("copy-data", "--source", SAVED_DATA, "--destination", DATA)
    return ["--data", DATA]

def run_training(command_name, model_name, run_record, *arguments):
    # Existing runs keep their settings. New runs use the profile from 00 or the training config.
    print(f"Reading saved settings for {model_name}…", flush=True)
    settings_file = run_record if run_record.is_file() else RUNS / f"{model_name}_profile.json"
    memory_options = []
    if settings_file.is_file():
        settings = json.loads(settings_file.read_text())["training"]
        memory_options = ["--microbatch-size", str(settings["microbatch_size"])]
        if settings["activation_checkpointing"]:
            memory_options.append("--activation-checkpointing")
    command = [sys.executable, "-u", "-m", "base_model_research.cli", command_name,
               "--model", f"configs/models/{model_name}.yaml", *map(str, arguments),
               *memory_options, "--progress-events"]
    print(f"Launching {model_name}. The update bar appears after the trainer checks "
          "saved runs and initializes the model.", flush=True)
    run_with_progress(command, cwd=REPO)


In [ ]:
FAMILY_RUNS = RUNS / "proxy" / "modified"

def run_proxy(*, first_only=False):
    options = ["--end", 1] if first_only else []
    run_training("sweep", "modified_proxy", FAMILY_RUNS / "protocol.json",
        *training_data(single_proxy_trial=first_only), "--output", RUNS / "proxy", *options)


In [ ]:
# Smoke run · train trial 000 on 20 tokens per parameter
if RUN_SMOKE:
    run_proxy(first_only=True)
    result = json.loads((FAMILY_RUNS / "trial_000" / "result.json").read_text())
    print(f"Trial 000 is {result['status']} at step {result['step']}.")
else:
    print("Smoke run disabled.")


In [ ]:
# Full run · all 32 trials
if RUN_FULL:
    run_proxy()
    winner = FAMILY_RUNS / "best_hparams.json"
    if winner.exists():
        best = json.loads(winner.read_text())
        print(f"Winner: {best['trial_id']} · validation loss {best['validation_loss']:.4f}")
        print(best["hyperparameters"])
    else:
        print("No winner yet. See", FAMILY_RUNS / "results.csv")
else:
    print("Full run disabled.")
